# Module 6: Prompting Techniques

## Learning Objectives
By the end of this module, you will be able to:
- Understand the principles of effective prompt engineering
- Apply zero-shot, few-shot, and chain-of-thought prompting
- Use self-consistency and tree of thoughts for complex reasoning
- Build prompt chains for multi-step workflows
- Use structured prompts for consistent outputs
- Debug and iterate on prompts for better results

---

## 1. What is Prompt Engineering?

**Prompt Engineering** is the art and science of crafting inputs to LLMs to get desired outputs.

### Why It Matters

```
Same Model + Different Prompts = Very Different Outputs
```

Unlike traditional programming where you write explicit code, with LLMs you **communicate intent through language**.

### The Prompt Engineering Mindset

| Traditional Programming | Prompt Engineering |
|------------------------|--------------------|
| Write explicit algorithms | Describe desired behavior |
| Debug code | Iterate on prompts |
| Compile errors are precise | LLM failures are subtle |
| One correct implementation | Many valid approaches |

---

## 2. Anatomy of a Good Prompt

### Key Components

```
┌────────────────────────────────────────────────────────────────┐
│  ROLE/PERSONA (optional)                                       │
│  "You are an expert Python developer..."                       │
├────────────────────────────────────────────────────────────────┤
│  CONTEXT                                                       │
│  Background information the model needs                        │
├────────────────────────────────────────────────────────────────┤
│  TASK/INSTRUCTION                                              │
│  What you want the model to do                                 │
├────────────────────────────────────────────────────────────────┤
│  FORMAT (optional)                                             │
│  How you want the output structured                            │
├────────────────────────────────────────────────────────────────┤
│  EXAMPLES (optional)                                           │
│  Demonstrations of desired behavior                            │
└────────────────────────────────────────────────────────────────┘
```

### Best Practices

1. **Be Specific**: Vague prompts get vague answers
2. **Provide Context**: Include relevant background
3. **Specify Format**: Tell the model how to structure output
4. **Use Delimiters**: Separate sections with `###`, `---`, or XML tags
5. **Iterate**: Refine based on results

In [ ]:
# Both Gemini (primary) and OpenAI (backup) use the same openai Python client
!uv pip install -q openai python-dotenv

In [ ]:
import os
from openai import OpenAI

# =============================================================
# PRIMARY: Gemini 2.0 Flash via OpenAI Compatibility
# Get your free key at: https://aistudio.google.com
# =============================================================

# Option 1 — Google Colab: Load API key from Colab Secrets
# from google.colab import userdata
# api_key = userdata.get('GEMINI_API_KEY')

# Option 2 — Local (VSCode / Jupyter): Load API key from .env file
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv("GEMINI_API_KEY")

api_key = os.getenv("GEMINI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# =============================================================
# BACKUP: OpenAI gpt-4o-mini
# Comment out the Gemini block above and uncomment below
# =============================================================
# api_key = os.getenv("OPENAI_API_KEY")
# client = OpenAI(api_key=api_key)

def chat(prompt, model="gemini-2.0-flash", temperature=0.7, system=None):
    """Helper to call the LLM API.

    Default: Gemini 2.0 Flash (free tier via Google AI Studio)
    Backup:  switch default to model="gpt-4o-mini" if using OpenAI
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print("✅ Gemini 2.0 Flash client ready! (via OpenAI compatibility)")

In [3]:
from IPython.display import display, Markdown

In [4]:
# Example: Bad vs Good Prompt

# ❌ Bad prompt - vague, no structure
bad_prompt = "Tell me about Python"

# ✅ Good prompt - specific, structured, clear format
good_prompt = """
Provide a brief overview of Python programming for a beginner.

Include:
1. What Python is (1 sentence)
2. Three main use cases
3. One simple code example

Keep the total response under 150 words.
"""

print("💬 Good Prompt Response:\n")
print(chat(good_prompt))

💬 Good Prompt Response:

Python is a high-level, versatile programming language known for its readability and ease of use. 

Three main use cases include:

1. **Web Development**: Using frameworks like Django and Flask to build dynamic websites.
2. **Data Science**: Analyzing data with libraries like Pandas and NumPy for statistical analysis and visualization.
3. **Automation/Scripting**: Writing scripts to automate repetitive tasks or manage system operations.

Here’s a simple code example that prints "Hello, World!":

```python
print("Hello, World!")
```

This code demonstrates Python's straightforward syntax, making it an excellent choice for beginners.


---

## 3. Zero-Shot Prompting

**Zero-shot** means asking the model to perform a task **without any examples** - relying purely on its pre-trained knowledge.

### When to Use
- Simple, well-defined tasks
- Standard formats the model has seen during training
- Quick prototyping

### Examples

In [3]:
# Zero-shot: Classification
zero_shot_classification = """
Classify the following customer review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The product arrived on time but the packaging was damaged. 
The item itself works fine though."

Classification:
"""

print("Classification result:")
print(chat(zero_shot_classification, temperature=0))

Classification result:
NEUTRAL


In [7]:
# Zero-shot: Summarization
zero_shot_summary = """
Summarize the following text in exactly 2 sentences:

---
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing algorithms that can access data and use it to learn 
for themselves. The process begins with observations or data, such as examples, 
direct experience, or instruction, to look for patterns in data and make better 
decisions in the future based on the examples provided.
---

Summary:
"""

print("Summary:")
Markdown(chat(zero_shot_summary, temperature=0))

Summary:


Machine learning, a subset of artificial intelligence, allows systems to learn and improve from experience without explicit programming by developing algorithms that analyze data. This process involves using observations or data to identify patterns and enhance decision-making based on past examples.

In [8]:
# Zero-shot: Code Generation
zero_shot_code = """
Write a Python function that:
1. Takes a list of numbers as input
2. Returns a dictionary with 'min', 'max', and 'average' keys
3. Include a docstring explaining the function

Only output the code, no explanations.
"""

print("Generated code:")
Markdown(chat(zero_shot_code, temperature=0))

Generated code:


```python
def calculate_statistics(numbers):
    """
    Calculate the minimum, maximum, and average of a list of numbers.

    Parameters:
    numbers (list): A list of numerical values.

    Returns:
    dict: A dictionary containing the minimum, maximum, and average of the input numbers.
    """
    if not numbers:
        return {'min': None, 'max': None, 'average': None}
    
    min_value = min(numbers)
    max_value = max(numbers)
    average_value = sum(numbers) / len(numbers)
    
    return {'min': min_value, 'max': max_value, 'average': average_value}
```

---

## 4. Few-Shot Prompting

**Few-shot** prompting provides **examples** of the desired input-output pattern. The model learns the pattern from examples and applies it to new inputs.

### When to Use
- Custom formats not standard in training data
- Specific output styles
- Domain-specific conventions
- Complex classification with edge cases

In [9]:
# Few-shot: Custom sentiment labeling with explanations
few_shot_sentiment = """
Classify the sentiment and provide a brief reason.

Example 1:
Text: "This laptop exceeds all my expectations!"
Sentiment: POSITIVE
Reason: Strong positive language ("exceeds expectations")

Example 2:
Text: "Decent product for the price point."
Sentiment: NEUTRAL
Reason: Balanced assessment, neither enthusiastic nor critical

Example 3:
Text: "Complete waste of money, broke after one week."
Sentiment: NEGATIVE
Reason: Strong negative language, product failure mentioned

Now classify:
Text: "It does what it's supposed to do, nothing more nothing less."
Sentiment:
Reason:
"""

print("Few-shot classification:")
Markdown(chat(few_shot_sentiment, temperature=0))

Few-shot classification:


Sentiment: NEUTRAL  
Reason: The statement indicates a functional but unremarkable performance, lacking enthusiasm or criticism.

In [10]:
# Few-shot: Custom data extraction format
few_shot_extraction = """
Extract structured information from job postings.

Example 1:
Input: "Senior Software Engineer at Google, Mountain View, CA. 
Requires 5+ years experience with Python and distributed systems. 
Salary range $180k-$250k."
Output:
- Title: Senior Software Engineer
- Company: Google
- Location: Mountain View, CA
- Experience: 5+ years
- Skills: Python, distributed systems
- Salary: $180k-$250k

Example 2:
Input: "Data Analyst role at Startup XYZ, remote position. 
Looking for someone with SQL expertise and 2-3 years in analytics."
Output:
- Title: Data Analyst
- Company: Startup XYZ
- Location: Remote
- Experience: 2-3 years
- Skills: SQL, analytics
- Salary: Not specified

Now extract:
Input: "AWS is hiring a Machine Learning Engineer in Seattle. 
Must have PhD or 7+ years ML experience. Strong TensorFlow and PyTorch skills required.
Competitive compensation with equity."
Output:
"""

print("Few-shot extraction:")
Markdown(chat(few_shot_extraction, temperature=0))

Few-shot extraction:


- Title: Machine Learning Engineer
- Company: AWS
- Location: Seattle
- Experience: PhD or 7+ years
- Skills: TensorFlow, PyTorch
- Salary: Competitive compensation with equity

---

## 5. Chain-of-Thought (CoT) Prompting

**Chain-of-thought** prompting asks the model to **show its reasoning step by step**, which significantly improves performance on complex tasks.

### Why It Works
- Forces the model to break down complex problems
- Reduces errors in multi-step reasoning
- Makes debugging easier (you can see where reasoning went wrong)

### Techniques
1. **Explicit CoT**: "Think step by step"
2. **Few-shot CoT**: Show examples with reasoning
3. **Self-Consistency**: Generate multiple chains, pick majority answer

In [11]:
# Without Chain-of-Thought (often fails on complex math)
no_cot = """
A store sells apples for $0.50 each. If you buy 5 or more, you get a 20% discount.
How much would 8 apples cost?
"""

# With Chain-of-Thought
with_cot = """
A store sells apples for $0.50 each. If you buy 5 or more, you get a 20% discount.
How much would 8 apples cost?

Think through this step by step:
1. Calculate the base price
2. Check if discount applies
3. Apply discount if applicable
4. State the final answer
"""

print("With Chain-of-Thought:")
Markdown(chat(with_cot, temperature=0))

With Chain-of-Thought:


Let's go through the steps to calculate the cost of 8 apples.

1. **Calculate the base price:**
   The price of one apple is $0.50. Therefore, the base price for 8 apples is:
   \[
   \text{Base Price} = 8 \times 0.50 = 4.00
   \]

2. **Check if discount applies:**
   The store offers a 20% discount if you buy 5 or more apples. Since we are buying 8 apples, the discount does apply.

3. **Apply discount if applicable:**
   To calculate the discount, we first find 20% of the base price:
   \[
   \text{Discount} = 0.20 \times 4.00 = 0.80
   \]
   Now, we subtract the discount from the base price:
   \[
   \text{Final Price} = 4.00 - 0.80 = 3.20
   \]

4. **State the final answer:**
   The cost of 8 apples after applying the discount is **$3.20**.

In [12]:
# Few-shot Chain-of-Thought for word problems
few_shot_cot = """
Solve the following problems by thinking step by step.

Problem: John has 3 times as many marbles as Sarah. Sarah has 7 marbles. 
How many marbles does John have?

Reasoning:
- Sarah has 7 marbles
- John has 3 times Sarah's amount
- John's marbles = 3 × 7 = 21
Answer: John has 21 marbles

---

Problem: A train travels at 60 mph. It needs to cover 150 miles. 
If it has already traveled for 1.5 hours, how much longer will it take?

Reasoning:
- Distance already covered = 60 mph × 1.5 hours = 90 miles
- Remaining distance = 150 - 90 = 60 miles
- Time needed for remaining = 60 miles ÷ 60 mph = 1 hour
Answer: 1 more hour

---

Problem: A rectangle's length is twice its width. If the perimeter is 36 cm, 
what are the dimensions?

Reasoning:
"""

print("Few-shot CoT:")
Markdown(chat(few_shot_cot, temperature=0))

Few-shot CoT:


Let's denote the width of the rectangle as \( w \) cm. Since the length is twice the width, we can express the length as \( 2w \) cm.

The formula for the perimeter \( P \) of a rectangle is given by:

\[
P = 2 \times (\text{length} + \text{width})
\]

Substituting the expressions for length and width into the perimeter formula, we have:

\[
36 = 2 \times (2w + w)
\]

Simplifying the equation:

\[
36 = 2 \times (3w)
\]
\[
36 = 6w
\]

Now, we can solve for \( w \):

\[
w = \frac{36}{6} = 6 \text{ cm}
\]

Now that we have the width, we can find the length:

\[
\text{length} = 2w = 2 \times 6 = 12 \text{ cm}
\]

Thus, the dimensions of the rectangle are:

- Width: 6 cm
- Length: 12 cm

Answer: Width is 6 cm and Length is 12 cm.

---

## 6. Structured Output Prompting

When you need outputs in specific formats (JSON, XML, tables), explicitly request the structure.

In [17]:
# JSON Output
json_prompt = """
Extract information from this text and return valid JSON:

"Meeting scheduled for December 20th at 2:30 PM in Conference Room B. 
Attendees: John Smith, Sarah Johnson, Mike Chen. 
Topic: Q4 Budget Review"

Return ONLY valid JSON with keys: date, time, location, attendees (array), topic
"""

result = chat(json_prompt, temperature=0)
print("JSON Output:")

import json
Markdown(result)

JSON Output:


```json
{
  "date": "December 20th",
  "time": "2:30 PM",
  "location": "Conference Room B",
  "attendees": ["John Smith", "Sarah Johnson", "Mike Chen"],
  "topic": "Q4 Budget Review"
}
```

In [18]:
# Markdown Table Output
table_prompt = """
Compare Python, JavaScript, and Go programming languages.

Create a markdown table with columns:
- Language
- Primary Use Case
- Typing System
- Learning Curve (Easy/Medium/Hard)

Return ONLY the markdown table.
"""

print("Table Output:")
Markdown(chat(table_prompt, temperature=0))

Table Output:


```markdown
| Language     | Primary Use Case               | Typing System | Learning Curve |
|--------------|--------------------------------|---------------|----------------|
| Python       | Web development, Data science  | Dynamic       | Easy           |
| JavaScript   | Web development, Frontend      | Dynamic       | Easy           |
| Go           | System programming, Cloud apps | Static        | Medium         |
```

---

## 7. Role/Persona Prompting

Assigning a **role or persona** can dramatically change the style and expertise level of responses.

In [20]:
# Same question, different personas
question = "What should I consider when designing a database schema?"

personas = {
    "Beginner-friendly teacher": 
        """You are a patient computer science teacher explaining to a first-year student. 
        Use simple analogies and avoid jargon.""",
    
    "Senior architect": 
        """You are a senior database architect at a Fortune 500 company. 
        Focus on enterprise-scale considerations and best practices.""",
    
    "Startup CTO":
        """You are a startup CTO who values speed and iteration. 
        Focus on practical trade-offs and avoiding over-engineering."""
}

for persona_name, persona_desc in personas.items():
    prompt = f"{persona_desc}\n\nQuestion: {question}\n\nAnswer (keep it concise, 3-4 sentences):"
    print(f"\n{'='*60}")
    print(f"Persona: {persona_name}")
    print(f"{'='*60}")
    print(chat(prompt, temperature=0.7))


Persona: Beginner-friendly teacher
When designing a database schema, think of it like creating a blueprint for a building. You need to decide what rooms (tables) you need, what furniture (data) will go in each room, and how these rooms will connect with each other (relationships). Consider what information is important and how people will use it, just like planning for how people will move through the building. Lastly, make sure it’s organized so that everything is easy to find and use, like a well-arranged closet.

Persona: Senior architect
When designing a database schema, consider normalization to eliminate data redundancy while ensuring efficient data retrieval through well-defined relationships. Prioritize scalability by anticipating future data growth and choosing appropriate data types and indexing strategies. Implement security measures, such as role-based access control, to safeguard sensitive information. Finally, ensure the schema supports robust backup and disaster recover

---

## 8. Advanced Techniques

### Self-Refinement
Ask the model to critique and improve its own output.

In [23]:
# Self-refinement prompting
self_refine = """
Task: Write a professional email declining a meeting invitation.

Step 1: Write a first draft.
Step 2: Critique your draft - what could be improved?
Step 3: Write an improved final version based on your critique.

Context: You're declining a meeting because of a schedule conflict, 
but want to suggest an alternative time.
"""

print("Self-refinement output:")
Markdown(chat(self_refine, temperature=0.7))

Self-refinement output:


**Step 1: First Draft**

Subject: Meeting Invitation

Dear [Recipient's Name],

Thank you for inviting me to the meeting on [date and time]. I appreciate the opportunity to discuss [meeting topic]. Unfortunately, I have a scheduling conflict at that time and will not be able to attend.

I would like to suggest rescheduling our meeting to [alternative date and time] if that works for everyone. I am looking forward to our discussion and hope we can find a time that accommodates all participants.

Thank you for your understanding.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]  

---

**Step 2: Critique of the Draft**

1. **Subject Line**: The subject line is generic and does not indicate that the email is about declining a meeting. A more specific subject line would be helpful.
  
2. **Tone**: While the tone is polite, it could be more considerate by expressing a regret for not being able to attend.

3. **Alternative Timing**: Instead of giving just one alternative, providing two or three options could be more accommodating.

4. **Closing Statement**: A more proactive closing could encourage further communication and express eagerness to connect.

---

**Step 3: Improved Final Version**

Subject: Regretfully Declining Meeting Invitation

Dear [Recipient's Name],

Thank you very much for inviting me to the meeting scheduled for [date and time]. I truly appreciate the opportunity to collaborate on [meeting topic]. However, I regret to inform you that I have a prior commitment and will be unable to attend.

To ensure I can contribute to our discussion, I would like to suggest a few alternative times: [Option 1: date and time], [Option 2: date and time], or [Option 3: date and time]. I hope one of these options works for everyone.

Thank you for your understanding, and I look forward to our conversation at a later date.

Warm regards,

[Your Name]  
[Your Position]  
[Your Contact Information]

### Constraint-Based Prompting
Explicitly list what the model should and shouldn't do.

In [24]:
# Constraint-based prompting
constrained = """
Write a product description for a new smartphone.

MUST:
- Be exactly 3 sentences
- Include one specific technical spec
- End with a call-to-action

MUST NOT:
- Use superlatives (best, greatest, etc.)
- Mention competitors
- Use exclamation marks

Product: Galaxy X20 with 200MP camera
"""

print("Constrained output:")
Markdown(chat(constrained, temperature=0.7))

Constrained output:


Introducing the Galaxy X20, a smartphone designed for those who appreciate high-quality photography with its impressive 200MP camera. Equipped with a powerful processor and a sleek, modern design, this device ensures smooth performance for all your daily tasks. Discover the art of mobile photography and elevate your experience by getting your Galaxy X20 today.

---

## 9. Self-Consistency with Chain-of-Thought

**Self-Consistency** extends CoT by generating **multiple reasoning paths** and selecting the most common answer (majority vote). This is particularly effective for problems where there may be multiple valid approaches.

### Why It Works
- Different reasoning paths may catch different errors
- Majority voting filters out outlier mistakes
- Increases reliability on complex problems

```
┌─────────────────────────────────────────────────────────────────┐
│                        Same Problem                             │
│                             │                                   │
│         ┌───────────────────┼───────────────────┐               │
│         ▼                   ▼                   ▼               │
│  ┌──────────────┐   ┌──────────────┐   ┌──────────────┐         │
│  │   Path 1     │   │   Path 2     │   │   Path 3     │         │
│  │  Answer: 42  │   │  Answer: 42  │   │  Answer: 40  │         │
│  └──────────────┘   └──────────────┘   └──────────────┘         │
│         │                   │                   │               │
│         └───────────────────┼───────────────────┘               │
│                             ▼                                   │
│                    Majority Vote: 42                            │
└─────────────────────────────────────────────────────────────────┘
```

In [5]:
# Self-Consistency: Generate multiple reasoning paths and take majority vote
from collections import Counter
import re

problem = """
A farmer has 15 sheep. All but 8 of them die. How many sheep are left?

Think through this step by step, then give your final answer as:
ANSWER: [number]
"""

# Generate multiple reasoning paths with temperature > 0 for diversity
n_samples = 5
responses = []
answers = []

print("Generating multiple reasoning paths...\n")
for i in range(n_samples):
    response = chat(problem, temperature=0.7)
    responses.append(response)
    
    # Extract the answer
    match = re.search(r'ANSWER:\s*(\d+)', response)
    if match:
        answers.append(int(match.group(1)))
    print(f"Path {i+1} answer extracted")

# Majority vote
if answers:
    vote_counts = Counter(answers)
    final_answer = vote_counts.most_common(1)[0][0]
    print(f"\n📊 Answers collected: {answers}")
    print(f"✅ Majority vote answer: {final_answer}")
else:
    print("Could not extract answers from responses")

Generating multiple reasoning paths...

Path 1 answer extracted
Path 2 answer extracted
Path 3 answer extracted
Path 4 answer extracted
Path 5 answer extracted

📊 Answers collected: [8, 8, 8, 8, 8]
✅ Majority vote answer: 8


---

## 10. Tree of Thoughts (ToT)

**Tree of Thoughts** extends Chain-of-Thought by exploring **multiple reasoning branches** at each step, evaluating them, and pursuing the most promising paths. This is powerful for complex problems requiring search and exploration.

### How It Differs from CoT

| Chain-of-Thought | Tree of Thoughts |
|------------------|------------------|
| Linear reasoning path | Branching exploration |
| Single attempt | Multiple candidates per step |
| No backtracking | Can prune bad branches |
| Fast | More thorough |

### When to Use ToT
- Creative problem solving
- Planning and strategy
- Problems with multiple valid approaches
- When you can afford more API calls for better results

In [8]:
# Tree of Thoughts: Explore multiple approaches, evaluate, and select best

problem = "Design a solution to reduce traffic congestion in a city center."

# Step 1: Generate multiple initial approaches
brainstorm_prompt = f"""
{problem}

Generate 3 different high-level approaches to solve this problem.
For each approach, give a brief 1-sentence description.

Format:
Approach 1: [description]
Approach 2: [description]
Approach 3: [description]
"""

print("Step 1: Brainstorming approaches...\n")
approaches = chat(brainstorm_prompt, temperature=0.8)
print(approaches)
print("\n" + "="*60)

Step 1: Brainstorming approaches...

Approach 1: Implement a comprehensive public transportation system that includes electric buses, trams, and bike-sharing programs to encourage residents and visitors to use alternative modes of transport instead of private vehicles.  

Approach 2: Introduce congestion pricing during peak hours, where drivers are charged a fee to enter high-traffic areas, thereby incentivizing off-peak travel and reducing the number of vehicles in the city center.  

Approach 3: Develop smart traffic management systems using AI and real-time data analysis to optimize traffic signals and manage flow, reducing bottlenecks and improving overall traffic conditions.  



In [9]:
# Step 2: Evaluate each approach
evaluate_prompt = f"""
Problem: {problem}

Here are three proposed approaches:
{approaches}

Evaluate each approach on these criteria (score 1-5):
- Feasibility: How realistic to implement?
- Impact: How much would it reduce congestion?
- Cost: How affordable? (5 = low cost)
- Timeline: How quickly can it be implemented? (5 = fast)

For each approach, give scores and a brief justification.
Then recommend which approach to pursue.
"""

print("Step 2: Evaluating approaches...\n")
evaluation = chat(evaluate_prompt, temperature=0.3)
print(evaluation)
print("\n" + "="*60)

🌳 Step 2: Evaluating approaches...

### Evaluation of Proposed Approaches

#### Approach 1: Comprehensive Public Transportation System
- **Feasibility: 3**
  - Implementing a comprehensive public transportation system requires significant planning, coordination, and investment. While feasible, it may face political and public resistance, especially regarding funding and changes to existing infrastructure.
  
- **Impact: 4**
  - A well-designed public transportation system can significantly reduce the number of private vehicles on the road, leading to a noticeable decrease in congestion. However, the extent of impact depends on the system's coverage and reliability.

- **Cost: 2**
  - Developing a comprehensive public transportation system is typically expensive due to the costs of infrastructure, vehicles, maintenance, and operational expenses.

- **Timeline: 2**
  - Designing and implementing a new public transportation system can take several years, if not decades, depending on the s

In [10]:
# Step 3: Develop the best approach in detail
develop_prompt = f"""
Based on this evaluation:
{evaluation}

Take the recommended approach and develop a detailed implementation plan with:
1. Specific actions (3-5 concrete steps)
2. Timeline for each step
3. Key stakeholders to involve
4. Potential risks and mitigations
"""

print("Step 3: Developing detailed plan...\n")
final_plan = chat(develop_prompt, temperature=0.5)
Markdown(final_plan)

Step 3: Developing detailed plan...



### Implementation Plan for Congestion Pricing

#### 1. Specific Actions

**Action 1: Conduct a Feasibility Study**
- **Description:** Assess the current traffic patterns, congestion levels, and potential economic impacts of implementing congestion pricing in the target areas. This study should also evaluate public attitudes towards congestion pricing and identify potential challenges.
- **Timeline:** 3 months
- **Key Stakeholders:** City traffic department, urban planners, local businesses, community organizations, transportation experts.

**Action 2: Develop a Pricing Structure and Technology Plan**
- **Description:** Create a detailed pricing structure (e.g., rates, time-of-day pricing) based on the findings from the feasibility study. Additionally, outline the technology needed for monitoring and collecting fees (e.g., cameras, sensors, payment systems).
- **Timeline:** 4 months
- **Key Stakeholders:** Traffic management authorities, technology vendors, financial analysts, urban planners.

**Action 3: Engage the Public and Stakeholders**
- **Description:** Implement a public engagement campaign to inform residents and stakeholders about the congestion pricing plan, its benefits, and how it will be implemented. Gather feedback and address concerns through town hall meetings, surveys, and informational sessions.
- **Timeline:** 3 months (concurrent with Action 2)
- **Key Stakeholders:** Community organizations, local government, businesses, residents, media.

**Action 4: Pilot Program Implementation**
- **Description:** Launch a pilot congestion pricing program in a selected area for a limited time to evaluate its effectiveness and gather data on traffic changes, public response, and revenue generation.
- **Timeline:** 6 months
- **Key Stakeholders:** Traffic management authorities, technology vendors, data analysts, local businesses.

**Action 5: Full-Scale Implementation and Monitoring**
- **Description:** Based on the pilot program's results, make necessary adjustments and roll out the full congestion pricing program across the designated areas. Establish ongoing monitoring and evaluation processes to assess the program's impact and make adjustments as needed.
- **Timeline:** 12 months
- **Key Stakeholders:** City traffic department, technology vendors, public transportation authorities, local government.

#### 2. Timeline Overview

| Action                                    | Duration       | Start Date | End Date   |
|-------------------------------------------|----------------|------------|------------|
| 1. Conduct a Feasibility Study           | 3 months       | Month 1    | Month 3    |
| 2. Develop a Pricing Structure and Tech Plan | 4 months       | Month 4    | Month 7    |
| 3. Engage the Public and Stakeholders    | 3 months       | Month 4    | Month 7    |
| 4. Pilot Program Implementation           | 6 months       | Month 8    | Month 13   |
| 5. Full-Scale Implementation and Monitoring| 12 months      | Month 14   | Month 25   |

#### 3. Key Stakeholders to Involve
- **City Traffic Department:** Responsible for traffic management and policy implementation.
- **Urban Planners:** Assist in designing the congestion pricing strategy and its integration with existing infrastructure.
- **Local Businesses:** Engage to understand their concerns and gather support for the program.
- **Community Organizations:** Help in reaching out to residents and gathering feedback.
- **Technology Vendors:** Provide the necessary technology solutions for monitoring and fee collection.
- **Public Transportation Authorities:** Collaborate to align congestion pricing with public transit improvements.

#### 4. Potential Risks and Mitigations

| Risk                                      | Mitigation Strategy                                   |
|-------------------------------------------|------------------------------------------------------|
| Public Opposition                         | Conduct thorough public engagement and education campaigns to address concerns and highlight benefits. |
| Technology Failures                       | Partner with experienced technology vendors and conduct rigorous testing before full implementation. |
| Economic Impact on Low-Income Residents   | Implement exemptions or discounts for low-income individuals and promote public transportation alternatives. |
| Insufficient Revenue Generation           | Continuously monitor and adjust pricing structures based on traffic patterns and revenue needs. |
| Ineffective Traffic Reduction             | Use data from the pilot program to refine the pricing structure and ensure it effectively encourages off-peak travel. |

### Conclusion
By following this detailed implementation plan, the city can effectively introduce congestion pricing as a means to reduce traffic congestion, enhance public transportation funding, and improve overall urban mobility. The structured approach ensures stakeholder involvement, risk management, and a focus on community benefits, leading to successful outcomes.

### Tree of Thoughts with Role-Based Instruction

Combining ToT with personas can be especially powerful - having an **expert** explore multiple reasoning paths brings domain knowledge to the exploration process.

In [11]:
# ToT + Role-Based: Expert explores multiple paths
expert_tot_prompt = """
You are a senior software architect with 20 years of experience designing scalable systems.

Problem: Design a real-time notification system for a social media app with 10 million users.

Think through this using Tree of Thoughts:

**Step 1: Generate 3 different architectural approaches**
For each, briefly describe the core pattern.

**Step 2: Evaluate each approach**
Consider: scalability, latency, cost, complexity

**Step 3: Select and detail the best approach**
Provide the high-level architecture with key components.

Think through each step carefully before moving to the next.
"""

print("Tree of Thoughts with Expert Persona:\n")
Markdown(chat(expert_tot_prompt, temperature=0.5))

Tree of Thoughts with Expert Persona:



### Step 1: Generate 3 Different Architectural Approaches

**Approach 1: Publish-Subscribe (Pub/Sub) Architecture**

**Core Pattern**: In this architecture, the notification system uses a message broker to handle notifications. When an event occurs (like a new message, comment, or like), the system publishes a message to a topic. Subscribers (users) who are interested in that topic receive the notifications in real-time.

**Approach 2: WebSocket-Based Push Notifications**

**Core Pattern**: This approach utilizes WebSocket connections to establish a persistent connection between the client and server. The server can push notifications to connected clients as soon as an event occurs, ensuring low-latency delivery.

**Approach 3: Server-Sent Events (SSE)**

**Core Pattern**: In this architecture, the server sends updates to the client over a single HTTP connection. Clients establish a connection to the server, which keeps the connection open and pushes notifications as they occur. This is simpler than WebSockets but is unidirectional (server to client only).

### Step 2: Evaluate Each Approach

**Approach 1: Publish-Subscribe (Pub/Sub) Architecture**

- **Scalability**: Highly scalable; can handle a large number of users and events due to decoupling of producers and consumers.
- **Latency**: Moderate latency due to the message broker, but can be optimized with proper configurations.
- **Cost**: Cost-effective if using managed services (e.g., AWS SNS, Google Pub/Sub) but can incur costs based on message volume.
- **Complexity**: Moderate complexity; requires additional infrastructure (message broker) and management of topics and subscriptions.

**Approach 2: WebSocket-Based Push Notifications**

- **Scalability**: Can be challenging to scale due to maintaining persistent connections for millions of users. Requires load balancing and horizontal scaling.
- **Latency**: Very low latency; notifications are pushed immediately to clients.
- **Cost**: Higher operational costs due to the need for maintaining persistent connections and resources.
- **Complexity**: More complex to implement and manage, especially concerning connection handling, reconnections, and scaling.

**Approach 3: Server-Sent Events (SSE)**

- **Scalability**: Less scalable than WebSockets for a large number of users due to the nature of HTTP connections.
- **Latency**: Low latency; notifications are pushed as they occur, but not as immediate as WebSockets for bi-directional communication.
- **Cost**: Moderate costs, as it uses standard HTTP connections, but can increase with high traffic.
- **Complexity**: Simpler to implement than WebSockets, but limited to unidirectional communication.

### Step 3: Select and Detail the Best Approach

**Selected Approach: Publish-Subscribe (Pub/Sub) Architecture**

#### High-Level Architecture

1. **Event Producer**:
   - Triggers notifications based on user actions (e.g., likes, comments, follows).
   - Sends messages to the message broker.

2. **Message Broker**:
   - A robust and scalable system (e.g., Apache Kafka, AWS SNS, RabbitMQ) that handles the distribution of messages to subscribers.
   - Manages topics for different types of notifications (e.g., user mentions, direct messages).

3. **Notification Service**:
   - Subscribes to relevant topics in the message broker.
   - Processes incoming messages and formats them for delivery to clients.
   - Can include logic for filtering notifications based on user preferences.

4. **User Notification Manager**:
   - Manages user subscriptions and preferences.
   - Keeps track of user sessions and their connection states.

5. **Client Application**:
   - Establishes a connection to the notification service using WebSockets or HTTP/2 for optimal performance.
   - Receives notifications in real-time and updates the user interface accordingly.

6. **Database**:
   - Stores user preferences, notification history, and other relevant data.
   - Can be a NoSQL database for scalability and performance.

#### Key Components

- **Load Balancer**: Distributes incoming requests to multiple instances of the Notification Service to ensure high availability and performance.
- **Monitoring and Logging**: Implement monitoring tools (e.g., Prometheus, Grafana) to track message delivery, latency, and system health.
- **Scaling Strategy**: Use auto-scaling for the notification service based on traffic patterns and user activity.

This architecture balances scalability, low latency, and cost-effectiveness while providing a robust foundation for real-time notifications in a social media app with a large user base.

---

## 11. Prompt Chaining (Multi-Step Prompting)

**Prompt Chaining** breaks complex tasks into a **sequence of simpler prompts**, where each step's output feeds into the next. This is essential for production applications.

### Benefits
- Easier to debug (pinpoint where things go wrong)
- More reliable (each step is simpler)
- More flexible (swap individual steps)
- Better for long tasks (avoids context window limits)

```
┌─────────────────────────────────────────────────────────────────┐
│         Prompt Chaining Pipeline                                │
│                                                                 │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐  │
│  │  Step 1  │───▶│  Step 2  │───▶│  Step 3  │───▶│  Step 4  │  │
│  │ Extract  │    │ Analyze  │    │ Generate │    │ Format   │  │
│  └──────────┘    └──────────┘    └──────────┘    └──────────┘  │
│       │              │               │               │         │
│       ▼              ▼               ▼               ▼         │
│   Raw Data      Insights       Draft Output     Final Output   │
└─────────────────────────────────────────────────────────────────┘
```

In [12]:
# Prompt Chaining: Multi-step document processing pipeline

document = """
TechCorp Q3 2024 Earnings Report Summary

Revenue reached $4.2 billion, up 18% year-over-year. Cloud services grew 35% to $1.8 billion, 
now representing 43% of total revenue. Hardware sales declined 5% to $1.1 billion due to 
supply chain constraints. Operating expenses increased 12% driven by AI research investments.
Net income was $680 million with earnings per share of $2.15.

CEO Jane Smith commented: "Our cloud transformation is accelerating. We expect cloud revenue 
to exceed hardware by Q2 2025. The $500M AI investment will position us for the next decade."

Guidance for Q4: Revenue of $4.4-4.6 billion with continued cloud growth momentum.
"""

# Step 1: Extract key metrics
step1_prompt = f"""
Extract all numerical metrics from this earnings report as JSON:

{document}

Return JSON with keys: total_revenue, cloud_revenue, hardware_revenue, 
yoy_growth, cloud_growth, net_income, eps, q4_guidance_low, q4_guidance_high
"""

print("Step 1: Extracting metrics...\n")
metrics = chat(step1_prompt, temperature=0)
print(metrics)

Step 1: Extracting metrics...

```json
{
  "total_revenue": 4200000000,
  "cloud_revenue": 1800000000,
  "hardware_revenue": 1100000000,
  "yoy_growth": 0.18,
  "cloud_growth": 0.35,
  "net_income": 680000000,
  "eps": 2.15,
  "q4_guidance_low": 4400000000,
  "q4_guidance_high": 4600000000
}
```


In [14]:
# Step 2: Analyze the data
step2_prompt = f"""
Analyze these financial metrics:
{metrics}

Provide:
1. Key strengths (2-3 bullets)
2. Key concerns (2-3 bullets)  
3. Overall assessment in one sentence
"""

print("\nStep 2: Analyzing data...\n")
analysis = chat(step2_prompt, temperature=0.3)
Markdown(analysis)


Step 2: Analyzing data...



### Key Strengths
- **Strong Revenue Growth**: The year-over-year (YoY) growth rate of 18% indicates a robust overall performance, suggesting effective business strategies and market demand.
- **Cloud Revenue Surge**: A significant 35% growth in cloud revenue highlights a successful transition towards high-margin services, positioning the company favorably in a competitive market.
- **Solid Profitability**: With a net income of $680 million and an earnings per share (EPS) of $2.15, the company demonstrates strong profitability, which can support future investments and shareholder returns.

### Key Concerns
- **Hardware Revenue Decline**: The hardware revenue of $1.1 billion may indicate a potential weakness in this segment, which could affect overall revenue diversification and stability.
- **Guidance Range**: The Q4 guidance of $4.4 billion to $4.6 billion suggests a potential slowdown or uncertainty in growth expectations, which could impact investor confidence.
- **Dependence on Cloud Growth**: While cloud revenue is growing rapidly, the company's reliance on this segment for future growth may pose risks if market conditions change or competition intensifies.

### Overall Assessment
The company shows strong overall growth and profitability, particularly in its cloud segment, but faces challenges in hardware revenue and future guidance that warrant careful monitoring.

In [15]:
# Step 3: Generate investor summary
step3_prompt = f"""
Based on this analysis:
{analysis}

Write a 3-sentence investor summary suitable for a financial newsletter.
Be objective and data-driven.
"""

print("\nStep 3: Generating summary...\n")
summary = chat(step3_prompt, temperature=0.5)
print(summary)


Step 3: Generating summary...

The company demonstrates strong year-over-year revenue growth of 18% and a notable 35% increase in cloud revenue, reflecting effective strategies and high market demand. However, a decline in hardware revenue to $1.1 billion raises concerns about revenue diversification, and the Q4 guidance of $4.4 billion to $4.6 billion may indicate potential growth uncertainty. Investors should be aware of the company's reliance on cloud growth, which, while currently robust, could be vulnerable to market fluctuations and competitive pressures.


In [16]:
# Step 4: Format final output
step4_prompt = f"""
Create a formatted report card:

Metrics: {metrics}
Analysis: {analysis}
Summary: {summary}

Format as a markdown report card with:
- Header with company name and quarter
- Key metrics table
- Strengths and concerns as bullet lists
- Executive summary at the bottom
"""

print("\nStep 4: Formatting final report...\n")
final_report = chat(step4_prompt, temperature=0.3)
Markdown(final_report)


📋 Step 4: Formatting final report...



# Company Name: [Your Company Name]
## Quarter: Q3 2023

### Key Metrics

| Metric                  | Value            |
|------------------------|------------------|
| Total Revenue          | $4,200,000,000   |
| Cloud Revenue          | $1,800,000,000   |
| Hardware Revenue       | $1,100,000,000   |
| Year-over-Year Growth   | 18%              |
| Cloud Growth           | 35%              |
| Net Income             | $680,000,000     |
| Earnings Per Share (EPS)| $2.15            |
| Q4 Guidance Low        | $4,400,000,000   |
| Q4 Guidance High       | $4,600,000,000   |

### Key Strengths
- **Strong Revenue Growth**: The year-over-year (YoY) growth rate of 18% indicates a robust overall performance, suggesting effective business strategies and market demand.
- **Cloud Revenue Surge**: A significant 35% growth in cloud revenue highlights a successful transition towards high-margin services, positioning the company favorably in a competitive market.
- **Solid Profitability**: With a net income of $680 million and an earnings per share (EPS) of $2.15, the company demonstrates strong profitability, which can support future investments and shareholder returns.

### Key Concerns
- **Hardware Revenue Decline**: The hardware revenue of $1.1 billion may indicate a potential weakness in this segment, which could affect overall revenue diversification and stability.
- **Guidance Range**: The Q4 guidance of $4.4 billion to $4.6 billion suggests a potential slowdown or uncertainty in growth expectations, which could impact investor confidence.
- **Dependence on Cloud Growth**: While cloud revenue is growing rapidly, the company's reliance on this segment for future growth may pose risks if market conditions change or competition intensifies.

### Overall Assessment
The company shows strong overall growth and profitability, particularly in its cloud segment, but faces challenges in hardware revenue and future guidance that warrant careful monitoring.

### Executive Summary
The company demonstrates strong year-over-year revenue growth of 18% and a notable 35% increase in cloud revenue, reflecting effective strategies and high market demand. However, a decline in hardware revenue to $1.1 billion raises concerns about revenue diversification, and the Q4 guidance of $4.4 billion to $4.6 billion may indicate potential growth uncertainty. Investors should be aware of the company's reliance on cloud growth, which, while currently robust, could be vulnerable to market fluctuations and competitive pressures.

### Chaining with Validation

In production, you can add validation between steps to catch errors early.

In [17]:
# Prompt chaining with validation
import json

def extract_with_validation(text):
    """Extract data with validation step"""
    
    # Step 1: Extract
    extract_prompt = f"""
    Extract the person's name and email from this text.
    Return ONLY valid JSON: {{"name": "...", "email": "..."}}
    
    Text: {text}
    """
    
    extraction = chat(extract_prompt, temperature=0)
    
    # Step 2: Validate
    try:
        # Try to parse JSON - handle markdown code blocks if present
        clean_json = extraction.strip()
        if clean_json.startswith('```'):
            clean_json = clean_json.split('```')[1]
            if clean_json.startswith('json'):
                clean_json = clean_json[4:]
        
        data = json.loads(clean_json)
        
        # Validate email format
        if '@' not in data.get('email', ''):
            return {"error": "Invalid email format", "raw": extraction}
            
        return {"success": True, "data": data}
        
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw": extraction}

# Test the validated chain
test_text = "Please contact John Doe at john.doe@techcorp.com for more info."
result = extract_with_validation(test_text)
print(f"Input: {test_text}")
print(f"Result: {result}")

Input: Please contact John Doe at john.doe@techcorp.com for more info.
Result: {'success': True, 'data': {'name': 'John Doe', 'email': 'john.doe@techcorp.com'}}


---

## 12. Common Prompting Mistakes

### ❌ Mistakes to Avoid

| Mistake | Problem | Better Approach |
|---------|---------|----------------|
| Too vague | "Help me with my code" | "Fix the TypeError in this Python function..." |
| Too long | 2000-word context | Focus on relevant info only |
| No format specified | Output is unpredictable | "Return as JSON with keys..." |
| Contradictory instructions | Confuses the model | Review for consistency |
| Assuming knowledge | "Use the standard approach" | Be explicit about what approach |

### ✅ Debugging Prompts

When outputs aren't what you expect:
1. **Simplify**: Remove complexity, test core instruction
2. **Add examples**: Show what you want
3. **Be more explicit**: Don't assume the model understands
4. **Check temperature**: Lower for consistency, higher for creativity
5. **Split complex tasks**: Chain multiple simpler prompts

---

## 📝 Student Exercises

### Exercise 1: Classification System
Create a few-shot prompt that classifies customer support tickets into categories: BILLING, TECHNICAL, GENERAL, URGENT.

In [ ]:
# Exercise 1: Create your classification prompt
classification_prompt = """
# TODO: Add your few-shot examples here
# Then classify this ticket:

"My account was charged twice for the same order. Please refund the extra charge."
"""

# print(chat(classification_prompt, temperature=0))

### Exercise 2: Chain-of-Thought for Code Review
Create a CoT prompt that analyzes code for bugs, explaining the reasoning.

In [ ]:
# Exercise 2: Create your code review prompt with CoT
code_to_review = """
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)
"""

review_prompt = f"""
# TODO: Create a prompt that:
# 1. Asks for step-by-step analysis
# 2. Identifies potential bugs
# 3. Suggests improvements

Code to review:
{code_to_review}
"""

# print(chat(review_prompt, temperature=0))

### Exercise 3: Structured Data Extraction
Create a prompt that extracts structured data from unstructured text.

In [ ]:
# Exercise 3: Extract structured data
text_to_extract = """
Dr. Sarah Chen joined Microsoft's AI Research division in Seattle 
as a Principal Researcher in March 2023. She previously worked at 
Stanford University for 8 years and has published over 50 papers 
on natural language processing.
"""

extraction_prompt = f"""
# TODO: Create a prompt that extracts:
# - Name, Title, Company, Location, Start Date
# - Previous employer, Years there
# - Research area, Publications count
# Return as JSON

Text:
{text_to_extract}
"""

# print(chat(extraction_prompt, temperature=0))

---

## 🎯 Key Takeaways

1. **Good prompts are specific, structured, and include format requirements**
2. **Zero-shot** works for standard tasks; **few-shot** teaches custom patterns
3. **Chain-of-thought** dramatically improves reasoning on complex tasks
4. **Self-consistency** increases reliability by generating multiple reasoning paths
5. **Tree of thoughts** explores multiple approaches for creative problem-solving
6. **Prompt chaining** breaks complex tasks into reliable, debuggable steps
7. **Personas/roles** change expertise level and communication style
8. **Iterate and debug** prompts like you would debug code

### Prompt Engineering Cheat Sheet

| Technique | Use When |
|-----------|----------|
| Zero-shot | Standard tasks, quick tests |
| Few-shot | Custom formats, domain-specific |
| Chain-of-thought | Math, logic, multi-step reasoning |
| Self-consistency | High-stakes decisions, complex reasoning |
| Tree of thoughts | Creative problems, planning, strategy |
| Prompt chaining | Production pipelines, long workflows |
| Role/Persona | Need specific expertise or tone |
| Structured output | Need JSON, tables, specific formats |
| Self-refinement | Quality is critical, time permits |

---

### Continue with:
- **02_function_calling.ipynb** - Connect LLMs to external tools
- **03_react_agent.ipynb** - Build reasoning agents

### Next Module: Fine-Tuning Pre-trained Models →